In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.layers import Dense, Dropout, Flatten,LSTM,GRU
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error ,mean_absolute_error

In [2]:
clns=["unit_number","time_cycles","op_setting_1","op_setting_2","op_setting_3"]+[f"sensor{i}" for i in range(1,22)]
fe=["op_setting_1","op_setting_2","op_setting_3"]+[f"sensor{i}" for i in range(1, 22)]

print(len(clns))

26


In [3]:
train_1=pd.read_csv("./nasa/train_FD001.txt",sep=r"\s+",header=None,names=clns)

In [4]:
test_1=pd.read_csv("./nasa/test_FD001.txt",sep=r"\s+",header=None,names=clns)

In [5]:
rul_1=pd.read_csv("./nasa/rul_FD001.txt",sep=r"\s+",header=None,names=["rul"])

In [6]:
mx_c=train_1.groupby("unit_number")["time_cycles"].transform("max")
train_1["rul"]=(mx_c-train_1["time_cycles"]).clip(upper=150)

In [7]:
sc=MinMaxScaler()
train_1[fe]=sc.fit_transform(train_1[fe])
test_1[fe]=sc.transform(test_1[fe])

In [8]:
uni=train_1["unit_number"].unique()
np.random.seed(42)
np.random.shuffle(uni)

n_train=int(len(uni)* 0.8)
train_uni=uni[:n_train]
val_uni=uni[n_train:]

In [9]:
s_l=30
xl=[]
yl=[]
for i in train_uni:
    en_data=train_1[train_1["unit_number"]==i].sort_values("time_cycles")
    data=en_data[fe].values
    rul_val=en_data["rul"].values
    for j in range(0,len(data)-s_l+1):
        wi=data[j:j+s_l]
        tar=rul_val[j+s_l-1]
        xl.append(wi)
        yl.append(tar)
x_train=np.array(xl)
y_train=np.array(yl)

In [10]:
xl_val,yl_val=[],[]
for i in val_uni:
    en_data=train_1[train_1["unit_number"]==i].sort_values("time_cycles")
    data=en_data[fe].values
    rul_val=en_data["rul"].values

    for j in range(len(data)-s_l+1):
        wi=data[j:j+s_l]
        tar=rul_val[j+s_l-1]
        xl_val.append(wi)
        yl_val.append(tar)

x_val=np.array(xl_val)
y_val=np.array(yl_val)

In [11]:
print(len(fe))

24


In [12]:
model=Sequential()
model.add(GRU(64,return_sequences=True,input_shape=(s_l,24)))
model.add(Dropout(0.2))
model.add(GRU(16,return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))

C:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [13]:
compile=model.compile(optimizer='adam',loss='mse',metrics=['mae'])

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 30, 64)         │        17,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 16)             │         3,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,505 (84.00 KB)

 Trainable params: 21,505 (84.00 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
sc2=MinMaxScaler()

y_train_sc=sc2.fit_transform(y_train.reshape(-1, 1))
y_val_sc=sc2.transform(y_val.reshape(-1, 1))

In [16]:
erl=EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True,verbose=1)
ckp=ModelCheckpoint("bst_modelFD001.keras",monitor='val_loss',save_best_only=True,verbose=1)

In [17]:
history=model.fit(x_train,y_train_sc,validation_data=(x_val,y_val_sc),epochs=100,batch_size=32,callbacks=[erl,ckp])

Epoch 1/100
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0748 - mae: 0.2090
Epoch 1: val_loss improved from None to 0.02929, saving model to bst_modelFD001.keras
439/439 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.0452 - mae: 0.1657 - val_loss: 0.0293 - val_mae: 0.1435
Epoch 2/100
436/439 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0279 - mae: 0.1324
Epoch 2: val_loss improved from 0.02929 to 0.01634, saving model to bst_modelFD001.keras
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.0240 - mae: 0.1219 - val_loss: 0.0163 - val_mae: 0.1000
Epoch 3/100
434/439 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0164 - mae: 0.1005
Epoch 3: val_loss did not improve from 0.01634
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.0154 - mae: 0.0975 - val_loss: 0.0206 - val_mae: 0.1150
Epoch 4/100
437/439 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.0144 - mae: 0.0934
Epoch 4: val_loss improved from 0.01634 to 0.01491, saving model to bst_modelFD001.keras
439/439 ━━━━━━━━━━━━━━━━━━━━ 13s 29m

In [18]:
l_model=load_model("bst_modelFD001.keras")
y_pre_sc=l_model.predict(x_val)
y_pred=sc2.inverse_transform(y_pre_sc)


116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [19]:
Rmse=np.sqrt(mean_squared_error(y_val,y_pred))
mae=mean_absolute_error(y_val,y_pred)

print(f"rmse: {Rmse:.2f}")
print(f"mae: {mae:.2f}")

rmse: 13.99
mae: 11.57


In [ ]:
def prepare_data(name,s_l=30,val_ratio=0.2,seed=42):
    clns=["unit_number","time_cycles","op_setting_1","op_setting_2","op_setting_3"]+[f"sensor{i}" for i in range(1,22)]
    fe=["op_setting_1","op_setting_2","op_setting_3"]+[f"sensor{i}" for i in range(1,22)]

    train=pd.read_csv(f"./nasa/train_{name}.txt",sep=r"\s+",header=None,names=clns)
    test=pd.read_csv(f"./nasa/test_{name}.txt",sep=r"\s+", header=None,names=clns)
    rul=pd.read_csv(f"./nasa/RUL_{name}.txt",sep=r"\s+",header=None, names=["rul"])

    mx_c=train.groupby("unit_number")["time_cycles"].transform("max")
    train["rul"]=mx_c-train["time_cycles"]

    sc=MinMaxScaler()
    train[fe]=sc.fit_transform(train[fe])
    test[fe]=sc.transform(test[fe])

    units=train["unit_number"].unique()
    np.random.seed(seed)
    np.random.shuffle(units)

    n_train=int(len(units)*(1-val_ratio))
    train_uni=units[:n_train]
    val_uni=units[n_train:]

    xl_train,yl_train=[],[]
    for i in train_uni:
        en_data=train[train["unit_number"]==i].sort_values("time_cycles")
        data=en_data[fe].values
        rul_val=en_data["rul"].values

        for j in range(len(data)-s_l+1):
            xl_train.append(data[j:j+s_l])
            yl_train.append(rul_val[j+s_l-1])

    x_train=np.array(xl_train)
    y_train=np.array(yl_train)

    xl_val,yl_val=[],[]
    for i in val_uni:
        en_data=train[train["unit_number"]==i].sort_values("time_cycles")
        data=en_data[fe].values
        rul_val=en_data["rul"].values

        for j in range(len(data)-s_l+1):
            xl_val.append(data[j:j+s_l])
            yl_val.append(rul_val[j+s_l-1])

    x_val=np.array(xl_val)
    y_val=np.array(yl_val)

    return x_train,y_train,x_val,y_val,test,rul,sc